In [0]:
dbutils.widgets.text("catalog", "claims")
catalog = dbutils.widgets.get("catalog")

In [0]:
# R5: claims.gold.claims_enriched — point-in-time gold, facts only
# Delta rejects a schema in CREATE OR REPLACE ... AS SELECT (RTAS), so: create with schema, then INSERT.
import pyspark.sql.functions as F

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

# R7: table is clustered by (admission_month, hospital_id) — liquid clustering, so
# future batches cluster incrementally without a full rewrite.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.gold.claims_enriched (
  claim_id STRING, member_id STRING, policy_id STRING, hospital_id STRING,
  admission_date DATE, discharge_date DATE, admission_month DATE,
  diagnosis_code STRING, procedure_code STRING, is_pre_existing BOOLEAN, claim_type STRING,
  billed_amount DECIMAL(12,2), room_rent_per_day DECIMAL(12,2), room_days INT,
  status STRING, approved_amount DECIMAL(12,2), rejection_code STRING,
  submitted_ts TIMESTAMP, settled_ts TIMESTAMP, channel STRING,
  plan_code_at_admission STRING, plan_family STRING, room_rent_cap_per_day INT,
  policy_start_date DATE, policy_tenure_months INT, length_of_stay INT,
  room_rent_over_cap BOOLEAN, turnaround_days INT, is_network_at_admission BOOLEAN,
  hospital_city STRING, hospital_city_tier STRING
) CLUSTER BY (admission_month, hospital_id)
""")

spark.sql(f"""
INSERT OVERWRITE {catalog}.gold.claims_enriched
WITH policy_versions AS (
  SELECT policy_id, plan_code, valid_from, valid_to FROM {catalog}.silver.dim_policy_scd2
),
policy_starts AS (
  SELECT policy_id, MIN(valid_from) AS policy_start_date
  FROM {catalog}.silver.dim_policy_scd2
  GROUP BY policy_id
),
c AS (
  SELECT
    claim_id, member_id, policy_id, hospital_id,
    CAST(admission_date AS DATE) AS admission_date,
    CAST(discharge_date AS DATE) AS discharge_date,
    date_trunc('month', CAST(admission_date AS DATE)) AS admission_month,
    diagnosis_code, procedure_code,
    CAST(is_pre_existing AS BOOLEAN) AS is_pre_existing, claim_type,
    CAST(billed_amount AS DECIMAL(12,2)) AS billed_amount,
    CAST(room_rent_per_day AS DECIMAL(12,2)) AS room_rent_per_day,
    CAST(room_days AS INT) AS room_days,
    status, CAST(approved_amount AS DECIMAL(12,2)) AS approved_amount,
    rejection_code,
    CAST(submitted_ts AS TIMESTAMP) AS submitted_ts,
    CAST(settled_ts AS TIMESTAMP) AS settled_ts,
    channel
  FROM {catalog}.silver.claims_clean
)
SELECT
  c.claim_id, c.member_id, c.policy_id, c.hospital_id,
  c.admission_date, c.discharge_date, c.admission_month,
  c.diagnosis_code, c.procedure_code, c.is_pre_existing, c.claim_type,
  c.billed_amount, c.room_rent_per_day, c.room_days,
  c.status, c.approved_amount, c.rejection_code,
  c.submitted_ts, c.settled_ts, c.channel,
  pv.plan_code AS plan_code_at_admission,
  pl.plan_family,
  pl.room_rent_cap_per_day,
  ps.policy_start_date,
  CAST(FLOOR(months_between(c.admission_date, ps.policy_start_date)) AS INT) AS policy_tenure_months,
  datediff(c.discharge_date, c.admission_date) AS length_of_stay,
  (c.room_rent_per_day > pl.room_rent_cap_per_day) AS room_rent_over_cap,
  CASE WHEN c.status = 'settled' AND c.settled_ts IS NOT NULL
       THEN CAST(datediff(CAST(c.settled_ts AS DATE), CAST(c.submitted_ts AS DATE)) AS INT) END AS turnaround_days,
  COALESCE(nh.hospital_id IS NOT NULL, FALSE) AS is_network_at_admission,
  ho.city AS hospital_city, ho.city_tier AS hospital_city_tier
FROM c
LEFT JOIN policy_versions pv
  ON c.policy_id = pv.policy_id
 AND c.admission_date >= pv.valid_from
 AND c.admission_date <= COALESCE(pv.valid_to, DATE'2099-12-31')
LEFT JOIN {catalog}.raw.plans pl ON pl.plan_code = pv.plan_code
LEFT JOIN {catalog}.raw.hospital_network_history nh
  ON nh.hospital_id = c.hospital_id
 AND c.admission_date >= nh.network_from
 AND c.admission_date <= COALESCE(nh.network_to, DATE'2099-12-31')
LEFT JOIN {catalog}.raw.hospitals ho ON ho.hospital_id = c.hospital_id
LEFT JOIN policy_starts ps ON ps.policy_id = c.policy_id
""")

print("claims_enriched rows:", spark.table(f"{catalog}.gold.claims_enriched").count())

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.gold.hospital_monthly (
  hospital_id STRING, admission_month DATE, claim_count INT,
  total_billed DECIMAL(12,2), total_approved DECIMAL(12,2), rejected_count INT)
""")
spark.sql(f"""
INSERT OVERWRITE {catalog}.gold.hospital_monthly
SELECT hospital_id, admission_month,
  CAST(COUNT(*) AS INT),
  CAST(ROUND(SUM(billed_amount), 2) AS DECIMAL(12,2)),
  CAST(ROUND(COALESCE(SUM(approved_amount), 0), 2) AS DECIMAL(12,2)),
  CAST(SUM(CASE WHEN status = 'rejected' THEN 1 ELSE 0 END) AS INT)
FROM {catalog}.gold.claims_enriched
GROUP BY hospital_id, admission_month
""")

In [0]:
# R5: verification — exact schema, forbidden columns absent, point-in-time invariants
import pyspark.sql.functions as F

ce = spark.table(f"{catalog}.gold.claims_enriched")
hm = spark.table(f"{catalog}.gold.hospital_monthly")

expected = ["claim_id","member_id","policy_id","hospital_id","admission_date","discharge_date","admission_month",
  "diagnosis_code","procedure_code","is_pre_existing","claim_type","billed_amount","room_rent_per_day","room_days",
  "status","approved_amount","rejection_code","submitted_ts","settled_ts","channel","plan_code_at_admission",
  "plan_family","room_rent_cap_per_day","policy_start_date","policy_tenure_months","length_of_stay",
  "room_rent_over_cap","turnaround_days","is_network_at_admission","hospital_city","hospital_city_tier"]
actual = [f.name for f in ce.schema.fields]
assert actual == expected, f"column mismatch: extra={set(actual)-set(expected)} missing={set(expected)-set(actual)}"

forbidden = ["payable","is_payable","payable_amount","eligible","is_eligible","waiting_period_ok","copay_pct","ped_waiting_months","initial_waiting_days"]
assert not [c for c in actual if c.lower() in forbidden], "forbidden verdict column present"

t = {f.name: f.dataType.simpleString() for f in ce.schema.fields}
assert t["admission_month"] == "date" and t["policy_start_date"] == "date"
assert t["plan_code_at_admission"] == "string" and t["policy_tenure_months"] == "int"
assert t["length_of_stay"] == "int" and t["turnaround_days"] == "int"
assert t["room_rent_over_cap"] == "boolean" and t["is_network_at_admission"] == "boolean"

assert ce.count() > 0
assert ce.groupBy("claim_id").count().filter("count > 1").count() == 0, "duplicate claim_id"
assert ce.filter("plan_code_at_admission IS NULL").count() == 0, "claim without plan at admission"
assert ce.filter("is_network_at_admission IS NULL").count() == 0, "is_network_at_admission must never be empty"
assert ce.filter("room_rent_over_cap IS NULL").count() == 0
assert ce.filter("hospital_city IS NULL").count() == 0
assert ce.filter("turnaround_days IS NOT NULL AND status <> 'settled'").count() == 0, "turnaround must be empty until settled"
# settled claims
assert ce.filter("policy_tenure_months < 0").count() == 0 and ce.filter("length_of_stay < 0").count() == 0

# hospital_monthly agrees exactly with a fresh recomputation from claims_enriched
recon = ce.groupBy("hospital_id", "admission_month").agg(
    F.count("*").alias("c"), F.round(F.sum("billed_amount"), 2).alias("b"),
    F.round(F.coalesce(F.sum("approved_amount"), F.lit(0)), 2).alias("a"),
    F.sum(F.when(F.col("status") == "rejected", 1).otherwise(0)).alias("r"))
mism = (hm.join(recon, ["hospital_id", "admission_month"])
        .filter((hm.claim_count != recon.c) | (hm.total_billed != recon.b) |
                (hm.total_approved != recon.a) | (hm.rejected_count != recon.r)))
assert mism.count() == 0, "hospital_monthly does not match claims_enriched"
assert hm.agg(F.sum("claim_count")).collect()[0][0] == ce.count(), "monthly claim_count does not sum to total claims"

print("All R5 checks passed:", ce.count(), "claims |", hm.count(), "hospital-month groups")

In [0]:
# R7: Delta internals — write history, time travel, clustering
# 1) The table's own write history
display(spark.sql("DESCRIBE HISTORY claims.gold.claims_enriched"))

In [0]:
# 2) Pull an earlier version on demand (disputed claim numbers weeks later)
# Version 1 = after the first INSERT (the build). Compare current vs v1 totals.
from pyspark.sql import functions as F

# Time travel is blocked on a table with a row filter (ROW_LEVEL_SECURITY_FEATURE_NOT_SUPPORTED.TIME_TRAVEL),
# so lift the R1 governance filter for this historical read, then restore it.
spark.sql("ALTER TABLE claims.gold.claims_enriched DROP ROW FILTER")
try:
    v1 = spark.read.format("delta").option("versionAsOf", 1).table("claims.gold.claims_enriched")
    cur = spark.table("claims.gold.claims_enriched")
    print("version 1 rows:", v1.count(), "| current rows:", cur.count())
    print("version 1 total billed:", v1.agg(F.round(F.sum("billed_amount"), 2)).collect()[0][0])
    print("current  total billed:", cur.agg(F.round(F.sum("billed_amount"), 2)).collect()[0][0])
finally:
    spark.sql("ALTER TABLE claims.gold.claims_enriched SET ROW FILTER claims.gov.claims_state_filter ON (hospital_id)")

In [0]:
# 3) Clustering proof: files organized by admission_month + hospital_id so monthly/hospital
# queries don't scan the whole table. Liquid clustering applies incrementally to new batches —
# no full rewrite needed (unlike static partitioning or manual OPTIMIZE-only clustering).
display(spark.sql("DESCRIBE DETAIL claims.gold.claims_enriched").select("clusteringColumns", "numFiles", "sizeInBytes"))